# elbo-loss-sum-with-beta — worked example 2: Full ELBO from raw VAE outputs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `elbo-loss-sum-with-beta`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In practice the two scalar terms are derived from the VAE forward pass: an MSE (or BCE) reconstruction between input and decoded output, and the closed-form diagonal-Gaussian KL from the encoder's `mu` and `logsigma`. The beta-weighted sum of these two batch-mean scalars is the loss that gets back-propagated.

## Worked solution

We assemble the loss end to end from a batch of inputs, reconstructions, and encoder statistics.

1. Reconstruction: square the per-element error `(x_hat - x) ** 2`, then take the mean over everything to get one scalar. This is the batch-mean MSE.
2. KL: the diagonal-Gaussian KL versus `N(0, I)` is `-0.5 * sum(1 + 2*logsigma - mu^2 - exp(2*logsigma))` per sample over the latent dimension. We sum over the latent axis to get a per-sample KL, then average over the batch for one scalar.
3. Combine: `recon + beta * kl`. The `beta` scales only the KL contribution, leaving the reconstruction term untouched.
4. We print the three numbers (recon, kl, total) so the relationship `total = recon + beta*kl` is visible.

In [ ]:
import torch as t

t.manual_seed(1)
x = t.randn(8, 16)
x_hat = x + 0.1 * t.randn(8, 16)
mu = t.randn(8, 4)
logsigma = t.randn(8, 4)

def vae_loss(x, x_hat, mu, logsigma, beta):
    recon = ((x_hat - x) ** 2).mean()
    kl = (-0.5 * (1 + 2 * logsigma - mu ** 2 - (2 * logsigma).exp()).sum(dim=1)).mean()
    return recon + beta * kl, recon, kl

total, recon, kl = vae_loss(x, x_hat, mu, logsigma, 0.5)
print('recon:', round(float(recon), 4), 'kl:', round(float(kl), 4), 'total:', round(float(total), 4))
print('reassembles:', bool(t.isclose(total, recon + 0.5 * kl)))